# StudyChat 실패 건만 재처리 — A100 80GB

기존 성공 결과를 보존하고 기존 full 결과에서 실패한 StudyChat만 재처리합니다.
**세션을 먼저 다시 시작한 뒤** 이 노트북을 위에서부터 실행하세요. 기존 ZIP은 Drive에서 재사용합니다.
기존 4·5·6번 셀을 실행하지 않습니다. 이 노트북에 전용 셀이 들어 있습니다.

기본값: 입력 131,072 / 출력 2,048 / 배치 8 / chunked prefill 16,384.
원본 대화는 자르지 않습니다. 한도를 넘는 입력은 계속 context_overflow로 남습니다.
A100에서는 공식 BF16 Qwen3.5-9B를 사용합니다. 기존 FP8 실행과 모델 정밀도가 달라
별도 폴더와 provenance.json에 기록합니다. 기존 ok 행은 교체하지 않습니다.
실제 A100 모델 로드·처리량은 미검증입니다. 배치 크기는 8배 속도를 보장하지 않습니다.


## 1. GPU 확인

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("런타임 → 런타임 유형 변경 → GPU로 설정한 뒤 다시 실행하세요.")
gpu = torch.cuda.get_device_properties(0)
print("GPU:", gpu.name)
print("VRAM GiB:", round(gpu.total_memory / 2**30, 2))
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)
if gpu.total_memory / 2**30 < 14:
    raise RuntimeError("이 노트북의 9B 시험 설정에는 14 GiB 이상 GPU를 권장합니다. 설정을 먼저 점검하세요.")


## 2. 기존 Drive·ZIP 연결

In [ ]:
# ruff: noqa: F821
import hashlib
import json
import shutil
import sys
import zipfile
from pathlib import Path

from google.colab import drive, files

drive.mount("/content/drive")
PROJECT_DIR = Path("/content/drive/MyDrive/ThinkGuard_NLP")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
RUN_NAME = "qwen35_9b_vllm_fp8_v1"  # 같은 설정으로 재개할 때는 바꾸지 마세요.
BUNDLE_NAME = "thinkguard_colab_vllm_bundle.zip"
BUNDLE = PROJECT_DIR / BUNDLE_NAME
WORK = Path("/content/thinkguard_colab_vllm")
WORK.mkdir(exist_ok=True)
if not BUNDLE.exists():
    print(f"제공받은 {BUNDLE_NAME} 하나를 업로드하세요.")
    uploaded = files.upload()
    if BUNDLE_NAME not in uploaded:
        raise ValueError("파일 이름을 확인하세요.")
    BUNDLE.write_bytes(uploaded[BUNDLE_NAME])
    del uploaded
shutil.copyfile(BUNDLE, WORK / "bundle.zip")
with zipfile.ZipFile(WORK / "bundle.zip") as archive:
    if archive.testzip() is not None:
        raise ValueError("ZIP checksum error")
    for name in archive.namelist():
        if Path(name).name != name or name.startswith("."):
            raise ValueError("Unexpected archive path")
    archive.extractall(WORK)
manifest = json.loads((WORK / "manifest.json").read_text(encoding="utf-8"))
for name, expected in manifest["files"].items():
    with (WORK / name).open("rb") as stream:
        observed = hashlib.file_digest(stream, "sha256").hexdigest()
    if observed != expected:
        raise ValueError(f"업로드 파일 검증 실패: {name}")
sys.path.insert(0, str(WORK))
print("원본 행 수:", manifest["counts"], "진단 표본:", manifest["references"])
print("Drive 저장 위치:", PROJECT_DIR)


## 3. 기존 버전 설치 — 재시작 안내 시 1번부터 다시 실행

In [ ]:
# ruff: noqa: F821
import importlib.metadata
import subprocess
import sys

packages = ["vllm", "torch", "transformers", "safetensors", "sentencepiece", "huggingface-hub"]
environment_file = PROJECT_DIR / "colab_vllm_package_versions.json"


def versions():
    result = {}
    for name in packages:
        try:
            result[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            result[name] = None
    return result


before = versions()
if environment_file.exists():
    previous = json.loads(environment_file.read_text(encoding="utf-8"))
    requirements = [f"{name}=={previous[name]}" for name in packages]
else:
    # Let the vLLM wheel select its matching PyTorch/CUDA dependencies.
    requirements = ["vllm>=0.17.0", "sentencepiece", "huggingface-hub"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *requirements])
installed = {name: importlib.metadata.version(name) for name in packages}
environment_file.write_text(json.dumps(installed, indent=2), encoding="utf-8")
print(installed)
restart_marker = WORK / "restart_required.json"
if before != installed:
    restart_marker.write_text(json.dumps({"pid": __import__("os").getpid()}), encoding="utf-8")
if restart_marker.exists():
    if json.loads(restart_marker.read_text())["pid"] == __import__("os").getpid():
        raise RuntimeError(
            "설치 완료. 런타임 → 세션 다시 시작 후 셀 1부터 다시 실행하세요. 지금 모델을 로드하지 마세요."
        )
    restart_marker.unlink()
print("vLLM 설치 확인 완료. 다음 셀을 실행하세요.")


## 4. 실패 건 선택 — 기존 성공 건은 제외

In [ ]:
"""Select failed StudyChat rows and preserve original successes during retry merge."""

from collections import Counter

from journal import record_key, stable_hash


def select_retry_sources(inputs, original, max_tokens):
    selected, seen = [], set()
    for source in inputs:
        key = record_key(source)
        previous = original.get(key)
        if previous is None or source["dataset"] != "studychat" or previous["status"] == "ok":
            continue
        if key in seen:
            raise ValueError("Duplicate retry source")
        seen.add(key)
        if any(previous.get(k) != source[k] for k in ("input_sha256", "source_sha256")):
            raise ValueError("Original result/input hash mismatch: " + key)
        if previous["status"] not in {"context_overflow", "invalid_output", "oom"}:
            continue
        selected.append(source)
    # Original counts are estimates for ordering only. Tokenizer checks the actual
    # current prompt (including repairs) before every engine call.
    selected.sort(key=lambda s: (original[record_key(s)].get("input_tokens", 0), record_key(s)))
    deferred = [record_key(s) for s in selected if original[record_key(s)].get("input_tokens", 0) > max_tokens]
    return selected, deferred


def merge_retry_results(original, retries, retry_config):
    merged = dict(original)
    recovered = 0
    for key, result in retries.items():
        previous = original.get(key)
        if key != record_key(result) or previous is None or previous["dataset"] != "studychat":
            raise ValueError("Unexpected retry result: " + key)
        if previous["status"] == "ok":
            raise ValueError("Retry attempted to replace original success: " + key)
        if result.get("run_signature") != stable_hash(retry_config):
            raise ValueError("Retry signature mismatch")
        if any(previous.get(k) != result.get(k) for k in ("input_sha256", "source_sha256")):
            raise ValueError("Retry input mismatch")
        if result["status"] == "ok":
            merged[key] = result
            recovered += 1
    report = {
        "processed": len(merged),
        "original_ok_preserved": sum(r["status"] == "ok" for r in original.values()),
        "recovered": recovered,
        "statuses": dict(Counter(r["status"] for r in merged.values())),
        "remaining_failed_ids": [k for k, r in merged.items() if r["status"] != "ok"],
        "note": "Mixed inference profiles; consult provenance.json. ok is validation, not human gold.",
    }
    return merged, report

import json
from collections import Counter
from journal import Journal, atomic_json, record_key, stable_hash
from runner import read_inputs, run, configuration, export_review

BASE_ROOT = PROJECT_DIR / "qwen35_9b_vllm_fp8_v1"
RETRY_ROOT = PROJECT_DIR / "qwen35_9b_a100_bf16_retry_v1"
RETRY_ROOT.mkdir(exist_ok=True)
base_config = json.loads((BASE_ROOT / "full/run_config.json").read_text(encoding="utf-8"))
if base_config["bundle_inputs_sha256"] != manifest["inputs_sha256"]:
    raise ValueError("Original result and input ZIP differ")
original_journal = Journal(BASE_ROOT / "full", base_config)
original = original_journal.records
MAX_INPUT = 131072
MAX_OUTPUT = 2048
BATCH_SIZE = 8  # 변경 시 RETRY_ROOT도 새 이름으로 변경하세요.
retry_sources, estimated_deferred = select_retry_sources(
    read_inputs(WORK / "inputs.jsonl.gz"), original, MAX_INPUT)
if not retry_sources:
    raise RuntimeError("재처리할 StudyChat 실패 건이 없습니다.")
print("기존 결과:", original_journal.summary())
print("실패 건만 선택:", len(retry_sources), "/ 기존 길이 기준 한도 초과:", len(estimated_deferred))
atomic_json(RETRY_ROOT / "selection.json", {"original_signature": stable_hash(base_config),
            "retry_ids": [record_key(s) for s in retry_sources], "estimated_deferred_ids": estimated_deferred})


## 5. A100 모델 로드 — 기존 모델 로드 셀 대신 이것만 실행

In [ ]:
import os
import time
import importlib.metadata
from functools import partial
from huggingface_hub import HfApi
import torch
from contract import chat_messages, parse_annotation

if "model" in globals():
    raise RuntimeError("이미 모델이 있습니다. 세션을 다시 시작하고 1번부터 실행하세요.")
if (WORK / "restart_required.json").exists():
    raise RuntimeError("설치 후 세션을 다시 시작하고 1번부터 실행하세요.")
if "A100" not in torch.cuda.get_device_name(0) or torch.cuda.get_device_properties(0).total_memory / 2**30 < 70:
    raise RuntimeError("이 설정은 A100 80GB용입니다. 할당된 GPU를 확인하세요.")
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
from vllm import LLM
MODEL_ID = "Qwen/Qwen3.5-9B"
pin_path = RETRY_ROOT / "model_revision.json"
if pin_path.exists():
    model_pin = json.loads(pin_path.read_text(encoding="utf-8"))
else:
    model_pin = {"model_id": MODEL_ID, "revision": HfApi().model_info(MODEL_ID).sha}
    atomic_json(pin_path, model_pin)
if model_pin["model_id"] != MODEL_ID:
    raise ValueError("Unexpected model pin")
settings = dict(dtype="bfloat16", max_model_len=MAX_INPUT + MAX_OUTPUT,
                max_num_seqs=BATCH_SIZE, max_num_batched_tokens=16384,
                gpu_memory_utilization=0.90, language_model_only=True,
                enable_chunked_prefill=True, enforce_eager=True, seed=0)
model = LLM(model=MODEL_ID, revision=model_pin["revision"], tokenizer_revision=model_pin["revision"],
            trust_remote_code=False, **settings)
tokenizer = model.get_tokenizer()
metadata = {"model_id": MODEL_ID, "revision": model_pin["revision"], "engine": "vllm",
            "quantization": "none; BF16", "engine_settings": settings,
            "versions": {p: importlib.metadata.version(p) for p in ("torch", "vllm", "transformers", "huggingface-hub")},
            "gpu": torch.cuda.get_device_name(0), "vram_gib": torch.cuda.get_device_properties(0).total_memory / 2**30}
print("로드 완료:", metadata)

class Annotator:
    batch_size = 8

    def __init__(self, model, tokenizer, max_input_tokens=6000, max_new_tokens=768):
        self.model, self.tokenizer = model, tokenizer
        self.max_input_tokens, self.max_new_tokens = max_input_tokens, max_new_tokens

    def __call__(self, source):
        return self.annotate_batch([source])[0]

    def annotate_batch(self, sources):
        from vllm import SamplingParams

        started = time.monotonic()
        results = [None] * len(sources)
        messages = [chat_messages(source) for source in sources]
        failures = [[] for _ in sources]
        active = list(range(len(sources)))
        for attempt in range(3):
            prompts, indices = [], []
            for i in active:
                if not sources[i].get("input_verified", True):
                    results[i] = {"status": "input_unverified", "error": "History structure did not validate"}
                    continue
                tokens = self.tokenizer.apply_chat_template(
                    messages[i], add_generation_prompt=True, tokenize=True, return_dict=False, enable_thinking=False
                )
                if not isinstance(tokens, list) or not tokens or any(type(t) is not int or t < 0 for t in tokens):
                    raise TypeError("Chat template must return a nonempty list of nonnegative integer token IDs")
                if len(tokens) > self.max_input_tokens:
                    results[i] = {
                        "status": "context_overflow",
                        "input_tokens": len(tokens),
                        "error": "Full input exceeds budget; no silent truncation",
                    }
                    continue
                prompts.append({"prompt_token_ids": tokens})
                indices.append(i)
            if not prompts:
                break
            # Engine failures abort the run; completed journal checkpoints remain resumable.
            outputs = self.model.generate(
                prompts, SamplingParams(temperature=0, max_tokens=self.max_new_tokens, seed=0), use_tqdm=False
            )
            if len(outputs) != len(indices):
                raise RuntimeError("vLLM returned an unexpected batch size")
            active = []
            for i, request, prompt in zip(indices, outputs, prompts, strict=False):
                completion = request.outputs[0]
                text = completion.text
                try:
                    if completion.finish_reason != "stop":
                        raise ValueError("Generation did not finish normally: " + str(completion.finish_reason))
                    parsed = parse_annotation(text, sources[i])
                    results[i] = {
                        **parsed,
                        "status": "ok",
                        "input_tokens": len(prompt["prompt_token_ids"]),
                        "output_tokens": len(completion.token_ids),
                        "attempts": attempt + 1,
                        "raw_response": text,
                        "validation_failures": list(failures[i]),
                    }
                except (ValueError, TypeError, KeyError, AttributeError) as exc:
                    failures[i].append(str(exc))
                    if attempt < 2:
                        messages[i] += [
                            {"role": "assistant", "content": text},
                            {
                                "role": "user",
                                "content": "Your annotation failed validation: "
                                + str(exc)
                                + ". Return complete JSON including all reasons and an evidence array (empty if appropriate). "
                            + "Copy short evidence quotes verbatim, including punctuation and whitespace, "
                            + "from a STUDENT turn only. Never paraphrase quotes or cite assistant turns. "
                            + "For current attempt=true cite turn=-1 and set cumulative=true. "
                            + "For past-only evidence establish the same task; otherwise use null if uncertain. "
                            + "Do not change judgments merely to pass validation. Keep reasons brief.",
                            },
                        ]
                        active.append(i)
                    else:
                        results[i] = {"status": "invalid_output", "error": "; ".join(failures[i])}
        elapsed = time.monotonic() - started
        for result in results:
            if result is None:
                raise RuntimeError("Missing batch result")
            result["batch_seconds"] = elapsed
            result["batch_size"] = len(sources)
        return results

annotator = Annotator(model, tokenizer, max_input_tokens=MAX_INPUT, max_new_tokens=MAX_OUTPUT)
annotator.batch_size = BATCH_SIZE
retry_config = configuration(manifest, metadata, annotator)
if any(retry_config[k] != base_config[k] for k in ("prompt_version", "prompt_sha256", "source_hashes")):
    raise ValueError("Original prompt/rules/source changed")
retry_config.update(retry_of=stable_hash(base_config), repair_version="exact-evidence-v1-three-attempts",
                    selection_sha256=stable_hash([record_key(s) for s in retry_sources]))
atomic_json(RETRY_ROOT / "current_hardware.json", metadata)
print("배치:", BATCH_SIZE, "입력:", MAX_INPUT, "출력:", MAX_OUTPUT)


## 6. 먼저 8건으로 엔진·출력 검증

In [ ]:
# 첫 실행은 8건만 검사하고, 재실행하면 저장된 행을 건너뜁니다.
retry_journal = run(retry_sources, annotator, RETRY_ROOT / "full", retry_config, limit=8)
first_results = list(retry_journal.records.values())[:8]
print("첫 처리 결과:", dict(Counter(r["status"] for r in first_results)))
print("원문/근거 검토:", export_review(retry_journal, retry_sources[:8]))


## 7. 실패 건 전체 재처리·재개

In [ ]:
# 첫 8건 상태와 근거를 확인한 뒤 이 셀을 실행하세요.
retry_journal = run(retry_sources, annotator, RETRY_ROOT / "full", retry_config)


## 8. 기존 성공 보존 후 병합·남은 실패 확인

In [ ]:
saved_retry = Journal(RETRY_ROOT / "full", retry_config)
merged, report = merge_retry_results(original, saved_retry.records, retry_config)
output_dir = RETRY_ROOT / "merged_owner_studychat"
output_dir.mkdir(exist_ok=True)
target = output_dir / "annotations.jsonl"
temp = target.with_suffix(".tmp")
with temp.open("w", encoding="utf-8") as stream:
    for key in sorted(merged):
        stream.write(json.dumps(merged[key], ensure_ascii=False) + "\n")
temp.replace(target)
atomic_json(output_dir / "coverage.json", report)
atomic_json(output_dir / "provenance.json", {"original_config": base_config, "retry_config": retry_config})
atomic_json(output_dir / "remaining_errors.json", [r for r in merged.values() if r["status"] != "ok"])
print({k: v for k, v in report.items() if k != "remaining_failed_ids"})
print("기존 성공 + 재처리 성공 병합:", target)
print("이 파일은 내 담당 범위입니다. 팀원 담당 StudyChat은 아직 포함되지 않습니다.")
